In [18]:
import numpy as np
import scipy.stats as stats
import src.noise as noise
import src.release as dprelease

from src.analysis import OddsRatio

Experimental parameters, no need to understand them at the outset.

In [19]:
# Standard deviation of Gaussian DP noise
NOISE_SCALE = 6.0

# Monte Carlo draws for CI estimation
MC_DRAWS = 2000

# Times to repeat the experiment
N_TRIALS = 500

Here's a contingency table.

In [20]:
true_tbl = np.array([
    [65, 109],
    [243, 1348]
])

It's straightforward to have `scipy` calculate the 95% confidence interval of its odds ratio.

In [21]:
scipy_ci = stats.contingency.odds_ratio(true_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (2.363633, 4.629785)


The `dpvalue` library can do this too. It implicitly converts `true_tbl` into a table with elements of type `NoisyFloat` (the noise is just zero).

In [22]:
dpvalue_lo, dpvalue_hi = OddsRatio(true_tbl).sample().confidence_interval(0.05)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (2.360639, 4.626609)


More or less consistent, I guess.

Let's add some noise and pretend the contingency table is a differentially private release.

In [23]:
noise_factory = noise.gaussian(loc=0.0, scale=NOISE_SCALE)
noisy_tbl = dprelease.noisy_float_array(true_tbl, noise_factory)
noisy_tbl

array([[NoisyFloat(obs=64.73375083749457, expr=id-961, thetas=frozenset({id-961}), eqns=(id-961 + id-960 - 64.7337508374946,)),
        NoisyFloat(obs=87.25937705632215, expr=id-963, thetas=frozenset({id-963}), eqns=(id-963 + id-962 - 87.2593770563222,))],
       [NoisyFloat(obs=242.66670488461335, expr=id-965, thetas=frozenset({id-965}), eqns=(id-965 + id-964 - 242.666704884613,)),
        NoisyFloat(obs=1353.2434989530173, expr=id-967, thetas=frozenset({id-967}), eqns=(id-967 + id-966 - 1353.24349895302,))]],
      dtype=object)

What would `scipy` do with this? Since `scipy` isn't DP-noise-aware, it can only take the observed values as truth and account for sampling uncertainty as it did before.

But oops, we can't use `scipy` on this directly, even though `NoisyFloat` is castable to `float`. The elements are counts, and have to be type `int`. Let's extract the noisy observations.

In [24]:
noisy_int_tbl = [
    [int(noisy_tbl[0, 0].obs), int(noisy_tbl[0, 1].obs)],
    [int(noisy_tbl[1, 0].obs), int(noisy_tbl[1, 1].obs)]]

noisy_int_tbl

[[64, 87], [242, 1353]]

And here's the `scipy` confidence interval again.

In [25]:
scipy_ci = stats.contingency.odds_ratio(noisy_int_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (2.896663, 5.839670)


But the `dpvalue` library can use `noisy_tbl` directly. It accounts for both uncertainty from DP noise _and_ uncertainty from sampling.

In [26]:
dpvalue_lo, dpvalue_hi = OddsRatio(noisy_tbl).sample().confidence_interval(0.05)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (2.692856, 6.325154)


And because of this, it knows the 95% odds ratio confidence interval for the DP release is actually a little bit wider.

How can we confirm if this interval is correct? Well, we know `true_tbl` odds ratio...

In [27]:
n0 = int(true_tbl[0, 0] + true_tbl[0, 1])
n1 = int(true_tbl[1, 0] + true_tbl[1, 1])
p0 = true_tbl[0, 0] / n0
p1 = true_tbl[1, 0] / n1

true_or = (p0 / (1.0 - p0)) / (p1 / (1.0 - p1))

We'll recreate `noisy_tbl` many times with fresh differential privacy noise draws _and_ fresh counts based on `n0`, `p0`, `n1`, and `p1`. We'll have `dpnoise` and `scipy` calculate 95% confidence intervals for each `noisy_tbl` we create. We'll count the number of times the confidence interval cover the true odds ratio.

In [28]:
dp_covers = 0
scipy_covers = 0

By definition of confidence interval, the intervals should contain the true odds ratio 95% of the time. We have to take it with a grain of salt though: the more trials we run and the more Monte Carlo samples we draw, the closer we get; but computational resources are limited.

In [29]:
for _ in range(N_TRIALS):
    # Step 1: simulate an ordinary (non-private) contingency table.
    grp0_yes = np.random.binomial(n0, p0)
    grp1_yes = np.random.binomial(n1, p1)
    true_tbl = np.array([
        [grp0_yes, n0 - grp0_yes],
        [grp1_yes, n1 - grp1_yes],
    ])

    # Differentially private release of true_tbl
    noisy_sim_tbl = dprelease.noisy_float_array(true_tbl, noise_factory)

    # Noise-aware CI (dpvalue)
    lo_dp, hi_dp = OddsRatio(noisy_sim_tbl).sample(n=MC_DRAWS).confidence_interval(0.05)
    dp_covers += (lo_dp <= true_or <= hi_dp)

    # Noise-unaware baseline: rounded noisy observations passed to scipy.
    noisy_int_tbl = np.rint([
        [noisy_sim_tbl[0, 0].obs, noisy_sim_tbl[0, 1].obs],
        [noisy_sim_tbl[1, 0].obs, noisy_sim_tbl[1, 1].obs],
    ]).astype(int)

    ci = stats.contingency.odds_ratio(noisy_int_tbl, kind="sample").confidence_interval(confidence_level=0.95)
    scipy_covers += (ci.low <= true_or <= ci.high)

In [30]:
print(f"Target odds ratio: {true_or:.4f}")
print(f"Replications: {N_TRIALS}, DP noise scale: {NOISE_SCALE}, MC draws/rep: {MC_DRAWS}")
print(f"dpvalue (noise-aware): {dp_covers/N_TRIALS:.3f}")
print(f"scipy on noisy counts (noise-unaware baseline): {scipy_covers/N_TRIALS:.3f}")

Target odds ratio: 3.3080
Replications: 500, DP noise scale: 6.0, MC draws/rep: 2000
dpvalue (noise-aware): 0.958
scipy on noisy counts (noise-unaware baseline): 0.912


So not exactly 0.95 for `dpvalue` but close enough to attribute the remainder to chance? The `scipy` confidence intervals are too narrow, they only contain the true value ~91% of the time. Probably more noise would amplify the difference.